# 공공데이터로 bar-chart-race 쇼츠 만들기: 세계 외환보유액 TOP 10 (1970~2025)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/team-datapopcorn/datapopcorn-assets/blob/main/bar-chart-race-shorts/bar_chart_race_shorts.ipynb)

세계은행 공개 API에서 나라별 외환보유액을 받아, 1위가 바뀌는 과정을 32초짜리 세로 영상(1080×1920, 30fps)으로 만듭니다. 한국은 파란 막대로 강조하고, 10위 밖일 때는 맨 아래 고정 줄에 순위를 보여 줍니다.

블로그 글: [datapopcorn.ai/blog/public-data-bar-chart-race-shorts](https://datapopcorn.ai/blog/public-data-bar-chart-race-shorts)

**실행 방법**: `런타임 → 모두 실행`. API 키, 로그인, GPU, Drive 연결은 필요 없습니다. 무료 런타임에서 약 2~3분 걸립니다.

| 단계 | 하는 일 |
|---|---|
| Step 1 | 세계은행 API에서 나라 목록과 지표를 받습니다 |
| Step 2 | 1위가 몇 번 바뀌는지 먼저 셉니다(긴장감 확인) |
| Step 3 | 빈 연도를 정해진 규칙으로만 채우고, 채운 비율을 기록합니다 |
| Step 4 | 프레임마다 TOP 12를 뽑고 연도 사이를 보간합니다 |
| Step 5 | Pillow로 1080×1920 프레임을 그려 ffmpeg로 바로 인코딩합니다 |
| Step 6 | 배경음악을 합치고, 프레임 수·전체 디코딩·모아보기로 검증합니다 |

In [ ]:
# Step 0. 준비: 한글 폰트(Pretendard)를 받습니다. ffmpeg는 Colab에 기본 설치돼 있습니다.
!mkdir -p fonts && cd fonts && for w in Regular Bold ExtraBold; do [ -f Pretendard-$w.otf ] || wget -q https://cdn.jsdelivr.net/gh/orioncactus/pretendard@v1.3.9/packages/pretendard/dist/public/static/Pretendard-$w.otf; done; ls
!ffmpeg -version | head -1

In [ ]:
import os, json, math, subprocess, urllib.request
import numpy as np, pandas as pd
from PIL import Image, ImageDraw, ImageFont

INDICATOR = "FI.RES.TOTL.CD"      # 외환보유액(금 포함, 명목 달러). 다른 지표 코드로 바꿔도 됩니다.
TITLE     = "외환보유액"
START, END = 1970, 2025
HERO      = "KOR"                 # 강조할 나라(ISO3)
MAX_FILL  = 2                     # 값이 빈 해는 최대 2년 전 값까지만 이어 씁니다
W, H, FPS, DUR = 1080, 1920, 30, 32
OUT = "out"; os.makedirs(OUT, exist_ok=True)

def get_json(url):
    req = urllib.request.Request(url, headers={"User-Agent": "datapopcorn-tutorial"})
    with urllib.request.urlopen(req, timeout=60) as r:
        return json.load(r)

## Step 1. 세계은행 API에서 데이터를 받습니다

세계은행 API는 지역 묶음(예: World, East Asia & Pacific)도 나라처럼 섞어서 돌려줍니다. 나라 목록에서 `region.id`가 `NA`인 행이 묶음이므로, 이것을 먼저 걸러야 '세계 1위'가 'World'가 되는 사고를 막을 수 있습니다.

In [ ]:
meta = get_json("https://api.worldbank.org/v2/country?format=json&per_page=400")[1]
countries = {c["id"]: c["name"] for c in meta if c["region"]["id"] != "NA"}
print("나라 수:", len(countries), "| 지역 묶음 제외:", len(meta) - len(countries))

rows = get_json(f"https://api.worldbank.org/v2/country/all/indicator/{INDICATOR}?format=json&per_page=20000&date={START}:{END}")[1]
df = pd.DataFrame([{"iso3": r["countryiso3code"], "year": int(r["date"]), "value": r["value"]} for r in rows])
df = df[df.iso3.isin(countries) & df.value.notna()]
raw = df.pivot(index="year", columns="iso3", values="value").reindex(range(START, END + 1))
print("관측값:", int(raw.notna().sum().sum()), "| 연도:", raw.index.min(), "~", raw.index.max())

## Step 2. 1위가 몇 번 바뀌는지 먼저 셉니다

bar-chart-race는 1위가 바뀌는 장면이 있어야 끝까지 봅니다. 영상을 만들기 전에 1위 교체 횟수를 세고, 2번 미만이면 다른 지표로 바꾸는 것을 기준으로 삼았습니다.

In [ ]:
leaders = raw.idxmax(axis=1).dropna()
changes = [(y, leaders[y]) for y in leaders.index[1:] if leaders[y] != leaders[y - 1]]
print("1위 교체", len(changes), "번:", changes)
assert len(changes) >= 2, "1위 교체가 2번 미만입니다. 다른 지표를 고르는 편이 낫습니다."

## Step 3. 빈 연도는 정해진 규칙으로만 채웁니다

나라마다 발표가 늦거나 빠진 해가 있습니다. 여기서는 **최대 2년 전 값까지만** 이어 쓰고, 그보다 오래 비면 빈칸으로 둡니다. 채운 칸이 얼마나 되는지 숫자로 남겨 두고, 영상 하단과 게시글에 이 규칙을 적습니다.

In [ ]:
filled = raw.ffill(limit=MAX_FILL)
added = int((filled.notna() & raw.isna()).sum().sum())
ranks = filled.rank(axis=1, ascending=False)
in_top = (ranks <= 10)
added_top = int((in_top & raw.isna()).sum().sum())
print(f"이어 쓴 칸: 전체 {added}개, 그중 TOP 10 안 {added_top}개 ({added_top / in_top.sum().sum():.1%})")

KO = {"USA":"미국","DEU":"독일","JPN":"일본","CHN":"중국","CHE":"스위스","SAU":"사우디","TWN":"대만","RUS":"러시아","IND":"인도",
      "KOR":"한국","HKG":"홍콩","SGP":"싱가포르","FRA":"프랑스","ITA":"이탈리아","GBR":"영국","BRA":"브라질","CAN":"캐나다","NLD":"네덜란드",
      "ESP":"스페인","MEX":"멕시코","THA":"태국","IRN":"이란","LBY":"리비아","VEN":"베네수엘라","KWT":"쿠웨이트","ARE":"UAE","NOR":"노르웨이",
      "BEL":"벨기에","AUT":"오스트리아","AUS":"호주","DZA":"알제리","NGA":"나이지리아","IDN":"인도네시아","MYS":"말레이시아","POL":"폴란드",
      "ISR":"이스라엘","CZE":"체코","TUR":"튀르키예","SWE":"스웨덴","PHL":"필리핀","IRQ":"이라크","QAT":"카타르","DNK":"덴마크","ZAF":"남아공",
      "ARG":"아르헨티나","CHL":"칠레","PRT":"포르투갈","GRC":"그리스","FIN":"핀란드","KAZ":"카자흐스탄","PER":"페루","EGY":"이집트","VNM":"베트남"}
name = lambda iso: KO.get(iso, countries.get(iso, iso))

## Step 4. 프레임마다 TOP 12를 뽑고 연도 사이를 보간합니다

화면에는 10개만 보이지만 11·12위까지 계산해 두면, 아래에서 올라오는 막대가 갑자기 튀어나오지 않습니다. 32초를 연도 수로 나눠 한 해에 약 0.5초가 돌아가고, 그 사이 값과 순위는 선형으로 보간합니다.

In [ ]:
years = list(filled.index)
N = FPS * DUR; RACE_END = DUR - 2.5                    # 마지막 2.5초는 최종 순위에서 멈춥니다
fmt = lambda v: f"{v/1e12:.2f}조 달러" if v >= 1e12 else f"{v/1e8:,.0f}억 달러"

def frame_state(t):
    q = min(max(t / RACE_END, 0), 1) * (len(years) - 1)
    i = min(int(q), len(years) - 2); p = q - i
    a, b = filled.loc[years[i]], filled.loc[years[i + 1]]
    v = (a.fillna(0) * (1 - p) + b.fillna(0) * p)
    v = v[(a.notna() | b.notna())].sort_values(ascending=False)
    return years[i] if p < 0.5 else years[i + 1], v

pos = {}                                               # 막대 위치를 부드럽게 옮기기 위한 이전 위치
def layout(v):
    out = []
    for r, (iso, val) in enumerate(v.items()):
        tgt = min(r, 11.5)
        cur = pos.get(iso, 12.5); cur += (tgt - cur) * 0.35
        pos[iso] = cur; out.append((iso, val, cur, r + 1))
    for iso in list(pos):
        if iso not in v.index: pos.pop(iso)
    return out

y0, v0 = frame_state(0); print(y0, [(name(k), fmt(x)) for k, x in v0.head(3).items()])
y1, v1 = frame_state(RACE_END); print(y1, [(name(k), fmt(x)) for k, x in v1.head(3).items()])

## Step 5. Pillow로 프레임을 그려 ffmpeg로 바로 인코딩합니다

PNG 960장을 저장했다가 합치는 대신, 그린 프레임을 원시 픽셀 그대로 ffmpeg 표준 입력에 흘려보냅니다. 디스크를 거의 쓰지 않고, 무료 런타임에서도 2분 안팎에 끝납니다. 막대와 숫자는 좌우 60px 안쪽, 화면 아래 약 330px 위에서 끝나게 그립니다. 인스타그램 릴스 UI가 하단과 오른쪽을 가리기 때문입니다.

In [ ]:
F = lambda w, s: ImageFont.truetype(f"fonts/Pretendard-{w}.otf", s)
BG, INK, MUTED, GOLD, GREY, BLUE = (5,7,11), (243,243,242), (188,196,206), (255,200,61), (94,104,118), (37,139,255)
X0, XMAX, Y0, STEP, BH = 130, 720, 600, 88, 64
def nice(x):
    m = 10 ** math.floor(math.log10(x))
    return next(k * m for k in (1, 1.2, 1.5, 2, 2.5, 3, 4, 5, 6, 8, 10) if k * m >= x)

def draw(t):
    year, v = frame_state(t); bars = layout(v)
    im = Image.new("RGB", (W, H), BG); d = ImageDraw.Draw(im)
    d.text((60, 200), f"세계은행 · {TITLE}(금 포함)", font=F("Bold", 28), fill=MUTED)
    d.text((60, 250), f"{TITLE} ", font=F("ExtraBold", 76), fill=INK)
    d.text((60 + d.textlength(f"{TITLE} ", font=F("ExtraBold", 76)), 250), "TOP 10", font=F("ExtraBold", 76), fill=GOLD)
    d.text((60, 360), str(year), font=F("ExtraBold", 110), fill=INK)
    kr = next((b for b in bars if b[0] == HERO), None)
    d.text((370, 380), "한국", font=F("Bold", 30), fill=MUTED)
    d.text((370, 420), f"세계 {kr[3]}위" if kr else "자료 없음", font=F("ExtraBold", 46), fill=(95,168,255))
    vmax = nice(v.iloc[0] * 1.05); vx = lambda x: X0 + x / vmax * (XMAX - X0)
    for k in (0, .5, 1):
        x = vx(vmax * k); d.line([(x, 560), (x, 1490)], fill=(37,41,48), width=2)
        d.text((x, 525), "0" if k == 0 else fmt(vmax * k).replace(" 달러", ""), font=F("Regular", 24), fill=MUTED, anchor="mm")
    for i in range(10): d.text((60, Y0 + i * STEP), f"{i+1:02d}", font=F("ExtraBold", 30), fill=GOLD if i == 0 else MUTED, anchor="lm")
    for iso, val, r, rank in sorted(bars, key=lambda b: -b[2]):
        if r > 9.6: continue
        y = Y0 + r * STEP; end = max(X0 + 8, vx(val)); col = BLUE if iso == HERO else GREY
        d.rounded_rectangle([X0, y - BH/2, end, y + BH/2], radius=6, fill=col)
        if iso == HERO: d.rounded_rectangle([X0, y - BH/2, end, y + BH/2], radius=6, outline=GOLD, width=5)
        nm = name(iso); fs = 32
        hide = any(o[0] != iso and abs(o[2] - r) < 0.6 and o[1] > val for o in bars)   # 자리를 바꾸는 중이면 아래 막대 숫자는 잠시 숨김
        if d.textlength(nm, font=F("Bold", fs)) < end - X0 - 24:
            d.text((X0 + 14, y), nm, font=F("Bold", fs), fill=INK, anchor="lm")
            if not hide: d.text((end + 14, y), fmt(val), font=F("Bold", 28), fill=INK, anchor="lm")
        elif not hide:
            d.text((end + 14, y), f"{fmt(val)}  {nm}", font=F("Bold", 28), fill=INK, anchor="lm")
    if kr and kr[3] > 10:                               # 한국이 10위 밖이면 고정 줄
        y = 1560; d.line([(60, 1505), (900, 1505)], fill=(52,57,65), width=2)
        d.text((60, y), str(kr[3]), font=F("ExtraBold", 30), fill=GOLD, anchor="lm")
        end = max(X0 + 120, vx(kr[1])); d.rounded_rectangle([X0, y - BH/2, end, y + BH/2], radius=6, fill=BLUE, outline=GOLD, width=5)
        d.text((X0 + 14, y), "한국", font=F("Bold", 32), fill=INK, anchor="lm"); d.text((end + 14, y), fmt(kr[1]), font=F("Bold", 28), fill=INK, anchor="lm")
    d.text((60, 1600), f"세계은행 WDI {INDICATOR} · 빈 해는 최대 {MAX_FILL}년 전 값 사용 · 연도 사이 보간", font=F("Regular", 20), fill=MUTED)
    d.rectangle([60, 1625, 900, 1629], fill=(52,57,65)); d.rectangle([60, 1625, 60 + 840 * min(t / RACE_END, 1), 1629], fill=GOLD)
    return im

draw(RACE_END * 0.6).resize((360, 640))

In [ ]:
silent = f"{OUT}/race_silent.mp4"; pos.clear()
enc = subprocess.Popen(["ffmpeg", "-y", "-loglevel", "error", "-f", "rawvideo", "-pix_fmt", "rgb24", "-s", f"{W}x{H}", "-r", str(FPS), "-i", "-",
                        "-c:v", "libx264", "-preset", "veryfast", "-crf", "18", "-pix_fmt", "yuv420p", "-movflags", "+faststart", silent], stdin=subprocess.PIPE)
for f in range(N):
    enc.stdin.write(draw(f / FPS).tobytes())
enc.stdin.close(); assert enc.wait() == 0
print("인코딩 완료:", silent, os.path.getsize(silent) // 1024, "KB")

## Step 6. 배경음악을 합치고 결과를 검증합니다

배경음악은 Kevin MacLeod의 "Club Diver"(incompetech.com, CC BY 4.0)를 씁니다. CC BY 곡이라 **게시글에 출처를 꼭 적어야 합니다.** 음악을 받지 못해도 무음 영상은 그대로 남습니다.

검증은 세 가지를 봅니다. (1) ffprobe로 영상 960프레임·해상도 확인, (2) 파일 전체를 끝까지 디코딩해 오류 0, (3) 시작·중간·끝 4장면을 모아 눈으로 확인합니다. 인코딩이 성공했다는 메시지만으로는 글자가 잘리거나 빈 화면인 것을 잡을 수 없기 때문입니다.

In [ ]:
final = f"{OUT}/race_final.mp4"
try:
    urllib.request.urlretrieve("https://incompetech.com/music/royalty-free/mp3-royaltyfree/Club%20Diver.mp3", "bgm.mp3")
    subprocess.run(["ffmpeg", "-y", "-loglevel", "error", "-i", silent, "-i", "bgm.mp3", "-map", "0:v", "-map", "1:a", "-c:v", "copy",
                    "-c:a", "aac", "-b:a", "192k", "-af", "afade=t=out:st=30:d=2", "-shortest", "-movflags", "+faststart", final], check=True)
    print("음악: Kevin MacLeod - Club Diver (incompetech.com, CC BY 4.0)")
except Exception as e:
    print("음악을 받지 못해 무음 영상을 최종본으로 씁니다:", e); final = silent

probe = subprocess.run(["ffprobe", "-v", "error", "-count_frames", "-show_entries", "stream=codec_type,width,height,nb_read_frames", "-of", "json", final],
                       capture_output=True, text=True)
print(json.dumps(json.loads(probe.stdout)["streams"], ensure_ascii=False))
err = subprocess.run(["ffmpeg", "-v", "error", "-i", final, "-f", "null", "-"], capture_output=True, text=True).stderr
print("전체 디코딩 오류:", len(err), "바이트")
assert len(err) == 0

In [ ]:
subprocess.run(["ffmpeg", "-y", "-v", "error", "-i", final, "-vf", "select='eq(n,0)+eq(n,300)+eq(n,600)+eq(n,955)',scale=270:480,tile=4x1",
                "-frames:v", "1", f"{OUT}/contact_sheet.jpg"], check=True)
from IPython.display import Image as IPImage, Video, display
display(IPImage(f"{OUT}/contact_sheet.jpg"))
Video(final, embed=True, width=300)

## 다른 지표로 바꿔 보기

맨 위 설정 셀에서 `INDICATOR`와 `TITLE`만 바꾸면 됩니다. 단위가 달러가 아니면 `fmt` 함수도 바꿉니다.

| 지표 | 코드 | 메모 |
|---|---|---|
| 상품·서비스 수출액 | `NE.EXP.GNFS.CD` | 1위 교체 3번 |
| 1인당 명목 GDP | `NY.GDP.PCAP.CD` | 초소형국이 1위를 차지하므로 인구 100만 명 이상만 남기는 규칙을 추가합니다 |
| 특허 출원(거주자) | `IP.PAT.RESD` | 1위 교체 1번뿐이라 Step 2에서 걸립니다 |

출처: World Bank, World Development Indicators(CC BY 4.0). 음악: Kevin MacLeod - Club Diver (incompetech.com, CC BY 4.0).